# Noisy simulations on MIMIQ
This page provide detailed information on simulating noise in quantum circuits using MIMIQ. 
A mathematical background is introduce first, then the function to add noise are presented. 

## Mathematical background

Noise in a quantum circuit refers to any kind of unwanted interaction of the qubits with the environment (or with itself). Mathematically, this puts us in the framework of open systems and the state of the qubits now needs to be described in terms of a density matrix $\rho$
, which fulfills $\rho$ = $\rho^\dagger$ and $Tr(\rho) = 1$.

A quantum operation such as noise can then be described using the Kraus operator representation as:
$$ \varepsilon(\rho) = \sum_k E_k \rho E_k^{\dagger}$$

We consider only completely positive and trace-preserving (CPTP) operations. In this case, the Kraus operators $E_k$
 can be any matrix as long as they fulfill the completeness relation:
 $$ \sum_k E_k E_k^{\dagger} = I$$

Note that unitary gates $U$ just correspond to a single Kraus operator $E_1 = U$

When all Kraus operators are proportional to a unitary matrix, $E_k = \alpha_k U_k $, this is called a mixed-unitary quantum operation and can be written as :
$$ \varepsilon(\rho) = \sum_k p_k U_k \rho U_k^{\dagger}$$

where $p_k = |\alpha_k|^2 $


## Evolution with noise
There are two common ways to evolve the state of the system when acting with Kraus channels : 
1. Density matrix
2. Quantum Trajectories : This method consists in simulating the evolution on the statevector $|\psi_{\alpha}>$ for a set of iterations $\alpha  = 1,2, .., n$. In each iteration, a noise channel is applied by randomly selecting one of the Kraus operators according to some probabilities (see below) and applying that Kraus operator to the state vector. The advantage of this approach is that we need less memory since we work with a state vector. The disadvantage is that we need to run the circuit many times to collect samples (one sample per run).
Currently, MIMIQ only implements is that a Kraus channel can be written as:
$$ \varepsilon(\rho) = \sum_k p_k \tilde{E_k} \rho \tilde{E_k}^{\dagger}$$
where $p_k = Tr(E_k \rho E_k^{\dagger})$ and $\tilde{E_k}/\sqrt{p_k} $

The parameter $p_k$ can be interpreted as probabilities since they fulfill $ 0 \leq p_k \leq 1$ and $\sum_k p_k = 1 $. In this way, the Kraus channel can be viewed as a linear combination of operatoions with differents Kraus operators weighted by the probabilities $p_k$.

In [35]:
from mimiqcircuits import *
import math

You can create noise channels using one of the many functions available. Most noise channels take one or more parameters, and custom channels require passing the Kraus matrices and/or probabilities. Here are some examples of how to build noise channels

### Building Noise channels
Creation of a noise channel. There are some example of how to create noise for a circuit. 

In [36]:
p = 0.1
PauliX(p)

PauliX(0.1)

In [37]:
p, gamma = 0.1, 0.2
GeneralizedAmplitudeDamping(p, gamma)

GeneralizedAmplitudeDamping((0.1, 0.2))

In [38]:
ps = [0.8, 0.1, 0.1]
paulis = ["II", "XX", "YY"]
PauliNoise(ps, paulis)

PauliNoise((0.8, pauli"II"), (0.1, pauli"XX"), (0.1, pauli"YY"))

In [39]:
from symengine import *
ps = [0.9, 0.1]
unitaries = [Matrix([[1, 0], [0, 1]]), Matrix([[1, 0], [0, -1]])] 
MixedUnitary(ps, unitaries)

MixedUnitary((0.9, "Custom([1 0; 0 1])"), (0.1, "Custom([1 0; 0 -1])"))

In [40]:
kmatrices = [Matrix([[1, 0], [0, (0.9)**0.5]]), Matrix([[0, (0.1)**0.5], [0, 0]])]    # Kraus matrices
Kraus(kmatrices)

Kraus(Operator([[1, 0], [0, 0.948683298050514]]), Operator([[0, 0.316227766016838], [0, 0]]))

In MIMIQ, the most important distinction between noise channels is whether they are mixed-unitary or general Kraus channels. You can use $\underline{ismixedunitary()}$ to check if a channel is mixed unitary

In [41]:
PauliX(p).ismixedunitary()

True

In [42]:
AmplitudeDamping(p).ismixedunitary()

False

 You can retriveve Kraus matrices/operators used to define a given channel using  krausmatrices() or krausoperators(). 

In [43]:
ProjectiveNoise("Z").krausmatrices()

[[1.0, 0]
 [0, 0],
 [0, 0]
 [0, 1.0]]

For mixed unitary channels, you can obtain the list of probabilities and the list of unitary gates/matrices separately using probabilities(), unitarymatrices(), or unitarygates(), respectively:

In [44]:
PauliZ(0.1).unitarymatrices()

[[1.0, 0]
 [0, 1.0],
 [1.0, 0]
 [0, -1.0]]

In [45]:
Depolarizing1(0.1).unitarygates()

[I, X, Y, Z]

In [46]:
PauliNoise([0.1, 0.9], ["II", "ZZ"])

PauliNoise((0.1, pauli"II"), (0.9, pauli"ZZ"))

 MIMIQ, noise channels can be added at any point in a circuit to make any operation noisy. For noisy gates, one would normally add a noise channel after an ideal gate. To model measurement, preparation, and reset errors, noise channels can be added before and/or after the corresponding operation. More information can be found in the next section

## How to add noise 

In [60]:
# Adding noise one by one
c = Circuit()
c.push(PauliX(0.1), [1,2,3,4,5]) # add noise
c.push(GateH(), 1)
c.push(AmplitudeDamping(0.1), 1) # add noise
c.push(GateCX(), 1, range(2,6))
c.push(PauliX(0.1), range(1,6)) # add noise
c.push(Measure(), range(1,6), range(1,6))
c.draw()

                                                                                
 q[0]: ╶───────────────────────────────────────────────────────────────────────╴
        ┌───────────┐                                                    ┌─┐    
 q[1]: ╶┤PauliX(0.1)├────────────────────────────────────────────────────┤H├───╴
        └───────────┘┌───────────┐                                       └─┘    
 q[2]: ╶─────────────┤PauliX(0.1)├─────────────────────────────────────────────╴
                     └───────────┘┌───────────┐                                 
 q[3]: ╶──────────────────────────┤PauliX(0.1)├────────────────────────────────╴
                                  └───────────┘┌───────────┐                    
 q[4]: ╶───────────────────────────────────────┤PauliX(0.1)├───────────────────╴
                                               └───────────┘┌───────────┐       
 q[5]: ╶────────────────────────────────────────────────────┤PauliX(0.1)├──────╴
                            

In [48]:
# Add noise to all gates of the same type
#Circuit().add_noise(gate, kraus, before=False, parallel==False)

In [61]:
cnoise = c.add_noise(Reset(), PauliX(0.1), parallel=True)
cnoise.add_noise(GateH(), AmplitudeDamping(0.1))
cnoise.draw()

                                                                                
 q[0]: ╶───────────────────────────────────────────────────────────────────────╴
        ┌───────────┐                                                    ┌─┐    
 q[1]: ╶┤PauliX(0.1)├────────────────────────────────────────────────────┤H├───╴
        └───────────┘┌───────────┐                                       └─┘    
 q[2]: ╶─────────────┤PauliX(0.1)├─────────────────────────────────────────────╴
                     └───────────┘┌───────────┐                                 
 q[3]: ╶──────────────────────────┤PauliX(0.1)├────────────────────────────────╴
                                  └───────────┘┌───────────┐                    
 q[4]: ╶───────────────────────────────────────┤PauliX(0.1)├───────────────────╴
                                               └───────────┘┌───────────┐       
 q[5]: ╶────────────────────────────────────────────────────┤PauliX(0.1)├──────╴
                            

In [62]:
cnoise.add_noise(GateCX(), Depolarizing2(0.1), parallel=True)


6-qubit, 6-bit circuit with 26 instructions:
├── PauliX(0.1) @ q[1]
├── PauliX(0.1) @ q[2]
├── PauliX(0.1) @ q[3]
├── PauliX(0.1) @ q[4]
├── PauliX(0.1) @ q[5]
├── H @ q[1]
├── AmplitudeDamping(0.1) @ q[1]
├── AmplitudeDamping(0.1) @ q[1]
├── CX @ q[1], q[2]
├── Depolarizing(0.1) @ q[1,2]
├── CX @ q[1], q[3]
├── Depolarizing(0.1) @ q[1,3]
├── CX @ q[1], q[4]
├── Depolarizing(0.1) @ q[1,4]
├── CX @ q[1], q[5]
├── Depolarizing(0.1) @ q[1,5]
├── PauliX(0.1) @ q[1]
├── PauliX(0.1) @ q[2]
├── PauliX(0.1) @ q[3]
⋮   ⋮
└── M @ q[5], c[5]

In [63]:
cnoise.add_noise(Measure(), PauliX(0.1), before=True, parallel=True)

6-qubit, 6-bit circuit with 31 instructions:
├── PauliX(0.1) @ q[1]
├── PauliX(0.1) @ q[2]
├── PauliX(0.1) @ q[3]
├── PauliX(0.1) @ q[4]
├── PauliX(0.1) @ q[5]
├── H @ q[1]
├── AmplitudeDamping(0.1) @ q[1]
├── AmplitudeDamping(0.1) @ q[1]
├── CX @ q[1], q[2]
├── Depolarizing(0.1) @ q[1,2]
├── CX @ q[1], q[3]
├── Depolarizing(0.1) @ q[1,3]
├── CX @ q[1], q[4]
├── Depolarizing(0.1) @ q[1,4]
├── CX @ q[1], q[5]
├── Depolarizing(0.1) @ q[1,5]
├── PauliX(0.1) @ q[1]
├── PauliX(0.1) @ q[2]
├── PauliX(0.1) @ q[3]
⋮   ⋮
└── M @ q[5], c[5]

In [64]:
cnoise.draw()

                                                                                
 q[0]: ╶───────────────────────────────────────────────────────────────────────╴
        ┌───────────┐                                                    ┌─┐    
 q[1]: ╶┤PauliX(0.1)├────────────────────────────────────────────────────┤H├───╴
        └───────────┘┌───────────┐                                       └─┘    
 q[2]: ╶─────────────┤PauliX(0.1)├─────────────────────────────────────────────╴
                     └───────────┘┌───────────┐                                 
 q[3]: ╶──────────────────────────┤PauliX(0.1)├────────────────────────────────╴
                                  └───────────┘┌───────────┐                    
 q[4]: ╶───────────────────────────────────────┤PauliX(0.1)├───────────────────╴
                                               └───────────┘┌───────────┐       
 q[5]: ╶────────────────────────────────────────────────────┤PauliX(0.1)├──────╴
                            

### Runnning a noisy channel

In [67]:
import random

rng = random.Random(42)

c = Circuit()
c.push(Depolarizing1(0.5), [1,2,3,4,5])
c.sample_mixedunitaries(rng=rng, ids=True)
c.draw()

                                                                                
 q[0]: ╶───────────────────────────────────────────────────────────────────────╴
        ┌─────────────────┐                                                     
 q[1]: ╶┤Depolarizing(0.5)├────────────────────────────────────────────────────╴
        └─────────────────┘┌─────────────────┐                                  
 q[2]: ╶───────────────────┤Depolarizing(0.5)├─────────────────────────────────╴
                           └─────────────────┘┌─────────────────┐               
 q[3]: ╶──────────────────────────────────────┤Depolarizing(0.5)├──────────────╴
                                              └─────────────────┘               
 q[4]: ╶───────────────────────────────────────────────────────────────────────╴
                                                                                
 q[5]: ╶───────────────────────────────────────────────────────────────────────╴
                            